# 04_01 — Limpieza, transformación y validación de NYC Taxi (2024)

**Proyecto:** Big Data con PySpark y Docker  
**Fuente principal:** NYC TLC — Yellow Taxi Trip Records (enero–diciembre de 2024)  
**Fase:** 04 — Limpieza, transformación e integración de datos

## Objetivo
Preparar un conjunto de datos **Silver**, trazable y reproducible, para analizar los viajes de taxi de 2024 y posteriormente integrarlos con las observaciones diarias de NOAA.

La estrategia consiste en **separar detección de anomalías y exclusión de registros**. La capa Raw permanece intacta; Silver conserva los viajes iniciados en 2024 y añade variables temporales y banderas de calidad. Se define también una selección *analítica* más estricta para movilidad, pero **no se eliminan de Silver** viajes de distancia cero, duraciones atípicas o importes negativos.

**Recorrido:** (1) entorno y lectura, (2) diagnóstico de calidad, (3) limpieza modular, (4) anomalías físicas y monetarias, (5) escritura incremental, (6) validación y promoción de Silver.

**Referencia de campos:** [TLC — diccionario Yellow Taxi](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf).

## 1. Entorno de ejecución

Se utiliza **PySpark 4.1.3** desde el contenedor Docker, con cuatro hilos locales (`local[4]`). La configuración de 24 particiones de *shuffle* es un punto de partida operativo, no un valor óptimo universal.

La lectura y escritura de Parquet se realizan sobre el volumen del repositorio montado en `/workspace`. Si se ejecuta el notebook fuera del contenedor, deben revisarse rutas, permisos y configuración de Java/Hadoop.

In [1]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# --------------------------------------------------
# 1. Localizar la raíz del proyecto
# --------------------------------------------------

def find_project_root(start: Path) -> Path:
    """Busca la raíz del repositorio a partir del directorio actual."""

    for candidate in [start, *start.parents]:
        if (
            (candidate / "data").is_dir()
            and (candidate / "notebooks").is_dir()
        ):
            return candidate

    raise FileNotFoundError("No se encontró la raíz del proyecto.")


PROJECT_ROOT = find_project_root(Path.cwd())

# --------------------------------------------------
# 2. Inicializar Spark
# --------------------------------------------------

spark = (
    SparkSession.builder
    .appName("04_01_taxi_data_cleaning")
    .master("local[4]")
    .config("spark.sql.shuffle.partitions", "24")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Raíz del proyecto:", PROJECT_ROOT)
print("Spark:", spark.version)

Raíz del proyecto: c:\Users\derec\Desktop\Proyectos\BigData_project
Spark: 4.1.3


In [2]:
import sys
import platform

print("Sistema operativo:", platform.system())
print("Python:", sys.executable)

Sistema operativo: Windows
Python: c:\Users\derec\anaconda3\envs\projects\python.exe


> **Nota sobre fechas:** Los timestamps de taxi corresponden al contexto horario de Nueva York. Antes de integrar NOAA o interpretar horas y duraciones cerca de cambios de horario de verano, debe comprobarse `spark.sql.session.timeZone` y la semántica de los timestamps leídos. No cambiamos la zona horaria en medio de esta ejecución porque invalidaría la comparabilidad de las métricas ya obtenidas.

## 2. Ingesta de datos Raw

Localizamos los doce Parquet mensuales, verificamos la cobertura enero–diciembre y conservamos la variable `source_month` para documentar **de qué archivo proviene cada fila**. Esta variable no debe confundirse con `pickup_month`, derivada de la fecha efectiva de inicio del viaje.

La lectura es **perezosa** (*lazy evaluation*): cargar el DataFrame y mostrar su esquema no materializa todos los registros.

In [3]:
# Localizar archivos originales de taxi correspondientes a 2024
taxi_files = sorted(
    (PROJECT_ROOT / "data").rglob(
        "yellow_tripdata_2024-??.parquet"
    )
)

print(f"Archivos encontrados: {len(taxi_files)}")

for path in taxi_files:
    print(path.relative_to(PROJECT_ROOT))

# Se esperan 12 archivos, uno por mes
assert len(taxi_files) == 12, (
    "Se esperaban 12 archivos Parquet de NYC Taxi para 2024."
)

# Lectura distribuida
df_raw = spark.read.parquet(
    *[str(path) for path in taxi_files]
)

# Extraer mes de procedencia del nombre del archivo
df_raw = df_raw.withColumn(
    "source_month",
    F.regexp_extract(
        F.input_file_name(),
        r"yellow_tripdata_(\d{4}-\d{2})\.parquet",
        1
    )
)

print("Columnas:", len(df_raw.columns))
df_raw.printSchema()

Archivos encontrados: 12
data\raw\taxi\year=2024\yellow_tripdata_2024-01.parquet
data\raw\taxi\year=2024\yellow_tripdata_2024-02.parquet
data\raw\taxi\year=2024\yellow_tripdata_2024-03.parquet
data\raw\taxi\year=2024\yellow_tripdata_2024-04.parquet
data\raw\taxi\year=2024\yellow_tripdata_2024-05.parquet
data\raw\taxi\year=2024\yellow_tripdata_2024-06.parquet
data\raw\taxi\year=2024\yellow_tripdata_2024-07.parquet
data\raw\taxi\year=2024\yellow_tripdata_2024-08.parquet
data\raw\taxi\year=2024\yellow_tripdata_2024-09.parquet
data\raw\taxi\year=2024\yellow_tripdata_2024-10.parquet
data\raw\taxi\year=2024\yellow_tripdata_2024-11.parquet
data\raw\taxi\year=2024\yellow_tripdata_2024-12.parquet
Columnas: 20
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = t

## 3. Diagnóstico inicial de calidad

### 3.1 Cobertura mensual, fechas, distancias y nulos

Se cuantifican inconsistencias a nivel de archivo de origen. Las categorías de error **pueden superponerse**: no deben sumarse para estimar un total de registros excluidos.

Se presta atención a las cinco columnas con nulos: `passenger_count`, `RatecodeID`, `store_and_fwd_flag`, `congestion_surcharge` y `Airport_fee`.

In [4]:
# Columnas que presentaron valores faltantes
null_columns = [
    "passenger_count",
    "RatecodeID",
    "store_and_fwd_flag",
    "congestion_surcharge",
    "Airport_fee"
]

# Expresiones de agregación
aggregations = [
    F.count("*").alias("total_records"),

    # Fechas fuera del año de análisis
    F.sum(
        F.when(
            F.year("tpep_pickup_datetime") != 2024,
            1
        ).otherwise(0)
    ).alias("outside_2024"),

    # Mes del viaje distinto al archivo de origen
    F.sum(
        F.when(
            F.date_format(
                "tpep_pickup_datetime", "yyyy-MM"
            ) != F.col("source_month"),
            1
        ).otherwise(0)
    ).alias("month_mismatch"),

    # Viajes con duración no positiva
    F.sum(
        F.when(
            F.col("tpep_dropoff_datetime")
            <= F.col("tpep_pickup_datetime"),
            1
        ).otherwise(0)
    ).alias("invalid_duration"),

    # Viajes con distancia no positiva
    F.sum(
        F.when(
            F.col("trip_distance") <= 0,
            1
        ).otherwise(0)
    ).alias("invalid_distance")
]

# Añadir conteos de nulos por columna
for column in null_columns:
    aggregations.append(
        F.sum(
            F.col(column).isNull().cast("long")
        ).alias(f"null_{column}")
    )

# Una agregación por archivo mensual
df_quality = (
    df_raw
    .groupBy("source_month")
    .agg(*aggregations)
    .orderBy("source_month")
)

df_quality.show(12, truncate=False)

+------------+-------------+------------+--------------+----------------+----------------+--------------------+---------------+-----------------------+-------------------------+----------------+
|source_month|total_records|outside_2024|month_mismatch|invalid_duration|invalid_distance|null_passenger_count|null_RatecodeID|null_store_and_fwd_flag|null_congestion_surcharge|null_Airport_fee|
+------------+-------------+------------+--------------+----------------+----------------+--------------------+---------------+-----------------------+-------------------------+----------------+
|2024-01     |2964624      |15          |18            |870             |60371           |140162              |140162         |140162                 |140162                   |140162          |
|2024-02     |3007526      |2           |15            |803             |68225           |185610              |185610         |185610                 |185610                   |185610          |
|2024-03     |3582628    

**Lectura de resultados**

- Se observaron **41,169,720** filas en los doce archivos y **56** viajes con fecha de inicio fuera de 2024.
- En **420** filas el mes de inicio difiere del mes del archivo. Esta discrepancia se registra, pero no es por sí misma motivo de exclusión: puede ocurrir cerca de los cambios de mes.
- Cada una de las cinco variables citadas contiene **4,091,232 nulos (9.94%)**, lo que exige estudiar conjuntamente su patrón.
- Se identificaron **13,510 duraciones no positivas** y **776,305 distancias no positivas**, antes de distinguir entre valores negativos y cero.

Por ahora **no se modifica ningún registro**.

### 3.2 Patrón conjunto de valores faltantes

Comprobamos si los cinco nulos ocurren en las mismas observaciones y su asociación con `VendorID` y `payment_type`. Es incorrecto asumir que los valores faltantes son completamente aleatorios (*MCAR*) sin estudiar este patrón.

In [5]:
# --------------------------------------------------
# 1. Identificar patrones de valores faltantes
# --------------------------------------------------

# Para cada columna creamos un indicador booleano:
# True  -> el valor está ausente
# False -> el valor está presente

null_flags = [
    F.col(column).isNull().alias(f"null_{column}")
    for column in null_columns
]

# Agrupar por combinaciones de valores faltantes
df_null_patterns = (
    df_raw
    .select(*null_flags)
    .groupBy(*[f"null_{c}" for c in null_columns])
    .count()
    .orderBy(F.desc("count"))
)

df_null_patterns.show(truncate=False)


# --------------------------------------------------
# 2. Analizar asociación con variables del viaje
# --------------------------------------------------

df_null_origin = (
    df_raw
    .withColumn(
        "has_null_passenger_count",
        F.col("passenger_count").isNull()
    )
    .groupBy(
        "VendorID",
        "payment_type",
        "has_null_passenger_count"
    )
    .count()
    .orderBy(F.desc("count"))
)

df_null_origin.show(30, truncate=False)

+--------------------+---------------+-----------------------+-------------------------+----------------+--------+
|null_passenger_count|null_RatecodeID|null_store_and_fwd_flag|null_congestion_surcharge|null_Airport_fee|count   |
+--------------------+---------------+-----------------------+-------------------------+----------------+--------+
|false               |false          |false                  |false                    |false           |37078488|
|true                |true           |true                   |true                     |true            |4091232 |
+--------------------+---------------+-----------------------+-------------------------+----------------+--------+

+--------+------------+------------------------+--------+
|VendorID|payment_type|has_null_passenger_count|count   |
+--------+------------+------------------------+--------+
|2       |1           |false                   |23071224|
|1       |1           |false                   |7380743 |
|2       |2        

**Hallazgo:** hay exactamente dos patrones: **37,078,488** filas con las cinco variables presentes y **4,091,232** con las cinco ausentes. En estos datos, las filas con los cinco nulos coinciden con `payment_type = 0`.

Según el diccionario TLC, `payment_type = 0` significa **Flex Fare**. La asociación observada justifica **conservar** estos registros y documentar el faltante, en vez de imputar pasajeros, suplementos u otros campos sin evidencia. El patrón no demuestra, por sí solo, por qué el sistema de captura omite esos valores.

### 3.3 Distancias y duraciones: ceros frente a negativos

Las distancias iguales a cero podrían corresponder a distintas situaciones operativas. Se clasifican por separado de las distancias negativas. También evaluamos la coincidencia entre distancias y duraciones no positivas para evitar decisiones de limpieza basadas en un único atributo.

In [6]:
# --------------------------------------------------
# 1. Calcular duración del viaje en segundos
# --------------------------------------------------

df_diagnostic = df_raw.withColumn(
    "duration_seconds",
    F.unix_timestamp("tpep_dropoff_datetime")
    - F.unix_timestamp("tpep_pickup_datetime")
)


# --------------------------------------------------
# 2. Clasificar anomalías
# --------------------------------------------------

df_anomalies = df_diagnostic.agg(

    # Distancias
    F.sum(
        (F.col("trip_distance") < 0).cast("long")
    ).alias("negative_distance"),

    F.sum(
        (F.col("trip_distance") == 0).cast("long")
    ).alias("zero_distance"),

    # Duraciones
    F.sum(
        (F.col("duration_seconds") < 0).cast("long")
    ).alias("negative_duration"),

    F.sum(
        (F.col("duration_seconds") == 0).cast("long")
    ).alias("zero_duration"),

    # Coincidencia entre ambos problemas
    F.sum(
        (
            (F.col("trip_distance") <= 0)
            & (F.col("duration_seconds") <= 0)
        ).cast("long")
    ).alias("both_invalid")
)

df_anomalies.show(truncate=False)

+-----------------+-------------+-----------------+-------------+------------+
|negative_distance|zero_distance|negative_duration|zero_duration|both_invalid|
+-----------------+-------------+-----------------+-------------+------------+
|0                |776305       |1575             |11935        |11124       |
+-----------------+-------------+-----------------+-------------+------------+



**Hallazgo:** no hubo distancias negativas; las **776,305** distancias no positivas fueron iguales a cero. Se identificaron **1,575** duraciones negativas y **11,935** duraciones cero. Un total de **11,124** registros presentó simultáneamente distancia y duración no positivas.

**Decisión:** marcar estas observaciones, sin eliminarlas de Silver. La selección para análisis de movilidad sí exigirá duración y distancia positivas.

## 4. Transformaciones de calidad reutilizables

La implementación que debe mantenerse a largo plazo reside en `src/etl/taxi_cleaning.py`. El notebook se centra en **orquestar, comprobar e interpretar** sus resultados en lugar de mantener una segunda versión extensa de la lógica de transformación.

**Funciones externas requeridas:**

- `prepare_taxi_data(df)`: fechas, duración, modalidad Flex Fare y banderas iniciales;
- `add_advanced_quality_flags(df)`: velocidad, anomalías monetarias y `is_mobility_core`;
- `filter_taxi_2024(df)`: selección de pickups de 2024.

El código modular requiere que el repositorio esté montado en `/workspace` y que `src/etl/taxi_cleaning.py` contenga esas funciones.

In [7]:
import sys
import importlib

# Permitir importar módulos del repositorio
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.etl.taxi_cleaning as taxi_cleaning

# Recargar el módulo durante el desarrollo del notebook
importlib.reload(taxi_cleaning)

# --------------------------------------------------
# 1. Aplicar reglas de calidad
# --------------------------------------------------

df_prepared = taxi_cleaning.prepare_taxi_data(df_raw)

# --------------------------------------------------
# 2. Construir dataset Silver
# --------------------------------------------------

df_silver = taxi_cleaning.filter_taxi_2024(df_prepared)

print("Transformaciones definidas correctamente.")

# Mostrar estructura de las nuevas variables
df_silver.select(
    "pickup_date",
    "pickup_hour",
    "duration_seconds",
    "is_flex_fare",
    "flag_invalid_duration",
    "flag_zero_distance",
    "is_candidate_mobility"
).show(5, truncate=False)

Transformaciones definidas correctamente.
+-----------+-----------+----------------+------------+---------------------+------------------+---------------------+
|pickup_date|pickup_hour|duration_seconds|is_flex_fare|flag_invalid_duration|flag_zero_distance|is_candidate_mobility|
+-----------+-----------+----------------+------------+---------------------+------------------+---------------------+
|2024-10-01 |0          |1062            |false       |false                |false             |true                 |
|2024-10-01 |0          |785             |false       |false                |false             |true                 |
|2024-10-01 |0          |546             |false       |false                |false             |true                 |
|2024-10-01 |0          |651             |false       |false                |false             |true                 |
|2024-10-01 |0          |17              |false       |false                |true              |false                |
+-----

### 4.1 Validación básica por modalidad

Los **indicadores de calidad** describen propiedades del registro; los de **elegibilidad** representan decisiones para casos de uso específicos. Por ejemplo, `is_candidate_mobility` exige fechas válidas, duración positiva, distancia positiva y ubicaciones presentes, pero los registros que no cumplen siguen en Silver.

A continuación se calcula el resumen a partir de la función modular, evitando duplicar manualmente su implementación.

In [8]:
# Resumen de calidad por modalidad de viaje

quality_flags = [
    "flag_out_of_year",
    "flag_month_mismatch",
    "flag_invalid_duration",
    "flag_zero_distance",
    "flag_missing_or_negative_distance",
    "flag_missing_location"
]

aggregations = [
    F.count("*").alias("total_records")
]

for flag in quality_flags:
    aggregations.append(
        F.sum(
            F.col(flag).cast("long")
        ).alias(flag)
    )

# Añadir conteos de elegibilidad
for indicator in [
    "is_eligible_for_daily_volume",
    "is_candidate_mobility"
]:
    aggregations.append(
        F.sum(
            F.col(indicator).cast("long")
        ).alias(indicator)
    )

df_quality_summary = (
    df_prepared
    .groupBy("is_flex_fare")
    .agg(*aggregations)
    .orderBy("is_flex_fare")
)

df_quality_summary.show(truncate=False)

+------------+-------------+----------------+-------------------+---------------------+------------------+---------------------------------+---------------------+----------------------------+---------------------+
|is_flex_fare|total_records|flag_out_of_year|flag_month_mismatch|flag_invalid_duration|flag_zero_distance|flag_missing_or_negative_distance|flag_missing_location|is_eligible_for_daily_volume|is_candidate_mobility|
+------------+-------------+----------------+-------------------+---------------------+------------------+---------------------------------+---------------------+----------------------------+---------------------+
|false       |37078488     |56              |420                |11916                |503616            |0                                |0                    |37078432                    |36573123             |
|true        |4091232      |0               |0                  |1594                 |272689            |0                                |0   

**Referencia del diagnóstico previo:** los registros con `is_candidate_mobility = True` sumaron **40,390,973** (36,573,123 tradicionales y 3,817,850 Flex Fare). No es un conteo de viajes definitivamente libres de valores atípicos, sino una primera selección orientada al análisis de distancia y duración.

### 4.2 Pruebas de regresión sobre el volumen 2024

Los siguientes `assert` contrastan la nueva implementación con métricas previamente verificadas. Funcionan como **pruebas de regresión** para *esta versión exacta* de los datos; no sustituyen pruebas unitarias con casos artificiales y resultados esperados.

In [9]:
# Resumen de validación del módulo ETL
df_validation = df_prepared.agg(

    F.count("*").alias("total_records"),

    F.sum(
        F.col("flag_out_of_year").cast("long")
    ).alias("out_of_year"),

    F.sum(
        F.col("is_eligible_for_daily_volume").cast("long")
    ).alias("valid_2024"),

    F.sum(
        F.col("is_candidate_mobility").cast("long")
    ).alias("mobility_candidates"),

    F.sum(
        F.col("flag_zero_distance").cast("long")
    ).alias("zero_distance"),

    F.sum(
        F.col("flag_invalid_duration").cast("long")
    ).alias("invalid_duration")
)

df_validation.show(truncate=False)

# Comprobar resultados esperados
result = df_validation.first()

assert result["total_records"] == 41_169_720
assert result["out_of_year"] == 56
assert result["valid_2024"] == 41_169_664
assert result["mobility_candidates"] == 40_390_973
assert result["zero_distance"] == 776_305
assert result["invalid_duration"] == 13_510

print("Todas las validaciones iniciales pasaron.")

+-------------+-----------+----------+-------------------+-------------+----------------+
|total_records|out_of_year|valid_2024|mobility_candidates|zero_distance|invalid_duration|
+-------------+-----------+----------+-------------------+-------------+----------------+
|41169720     |56         |41169664  |40390973           |776305       |13510           |
+-------------+-----------+----------+-------------------+-------------+----------------+

Todas las validaciones iniciales pasaron.


## 5. Anomalías físicas, monetarias y geográficas

Se calcula la velocidad promedio para viajes candidatos a movilidad:

\[
 v_i = \frac{d_i}{t_i}\,3600,
\]

donde `d_i` es la distancia reportada en **millas**, `t_i` es la duración en **segundos** y `v_i` se expresa en **mph**.

Se estudian velocidades superiores a **80 mph** y duraciones superiores a **4 horas** como **umbrales exploratorios**; no son reglas físicas universales. Los importes negativos se etiquetan, dado que pueden corresponder a ajustes contables u otros registros que requieren contexto. Los IDs de ubicación positivos no garantizan pertenencia al catálogo oficial de zonas TLC.

In [10]:
# --------------------------------------------------
# 1. Calcular velocidad promedio
# --------------------------------------------------

df_diagnostic = df_prepared.withColumn(
    "speed_mph",
    F.when(
        F.col("is_candidate_mobility"),
        F.col("trip_distance") * 3600
        / F.col("duration_seconds")
    )
)

# --------------------------------------------------
# 2. Función auxiliar para contar condiciones
# --------------------------------------------------

def count_if(condition, alias):
    """
    Cuenta registros que cumplen una condición.

    Los valores NULL no se contabilizan como True.
    """
    return F.sum(
        F.when(condition, 1).otherwise(0)
    ).alias(alias)


# --------------------------------------------------
# 3. Diagnóstico por modalidad
# --------------------------------------------------

df_extremes = (
    df_diagnostic
    .groupBy("is_flex_fare")
    .agg(
        F.count("*").alias("total_records"),

        # Velocidades superiores a 80 mph
        count_if(
            F.col("speed_mph") > 80,
            "speed_over_80_mph"
        ),

        # Viajes con duración superior a 4 horas
        count_if(
            F.col("duration_seconds") > 4 * 3600,
            "duration_over_4h"
        ),

        # Importes monetarios negativos
        count_if(
            F.col("fare_amount") < 0,
            "negative_fare"
        ),

        count_if(
            F.col("total_amount") < 0,
            "negative_total"
        ),

        count_if(
            F.col("tip_amount") < 0,
            "negative_tip"
        ),

        # Códigos geográficos no positivos
        count_if(
            (F.col("PULocationID") <= 0)
            | (F.col("DOLocationID") <= 0),
            "invalid_location_id"
        ),

        # Percentiles de velocidad
        F.percentile_approx(
            "speed_mph",
            [0.50, 0.95, 0.99, 0.999],
            10000
        ).alias("speed_percentiles"),

        # Velocidad máxima
        F.max("speed_mph").alias("max_speed_mph")
    )
    .orderBy("is_flex_fare")
)

df_extremes.show(truncate=False)

+------------+-------------+-----------------+----------------+-------------+--------------+------------+-------------------+------------------------------------------------------------------------------+-----------------+
|is_flex_fare|total_records|speed_over_80_mph|duration_over_4h|negative_fare|negative_total|negative_tip|invalid_location_id|speed_percentiles                                                             |max_speed_mph    |
+------------+-------------+-----------------+----------------+-------------+--------------+------------+-------------------+------------------------------------------------------------------------------+-----------------+
|false       |37078488     |11486            |23227           |603932       |606791        |1330        |0                  |[9.203007518796992, 24.445868033311978, 35.201866977829646, 47.75675675675676]|807742.4299065421|
|true        |4091232      |1758             |147             |127092       |2553          |1           |0  

**Interpretación del diagnóstico:**

| Indicador | Tradicional | Flex Fare | Total |
|:--|--:|--:|--:|
| Velocidad >80 mph | 11,486 | 1,758 | 13,244 |
| Duración >4 horas | 23,227 | 147 | 23,374 |
| `fare_amount` negativo | 603,932 | 127,092 | 731,024 |
| `total_amount` negativo | 606,791 | 2,553 | 609,344 |
| `tip_amount` negativo | 1,330 | 1 | 1,331 |
| Zona de origen/destino no positiva | 0 | 0 | 0 |

Aunque los máximos de velocidad son físicamente inverosímiles, la distribución central es razonable (medianas de **9.20** y **10 mph** para tradicionales y Flex Fare, respectivamente). Esto favorece **crear banderas** y diferenciar aplicaciones analíticas en lugar de borrar indiscriminadamente filas.

Un importe negativo **no prueba** que el viaje sea inválido: el sentido contable del registro no se resuelve solo con la señal del monto.

### 5.1 Aplicación de reglas avanzadas

Se incorporan `speed_mph`, las banderas de velocidad y duración extremas, las alertas monetarias y `is_mobility_core` (subconjunto orientado a análisis de movilidad). Las reglas no modifican los importes ni realizan imputaciones.

In [11]:
# Recargar las modificaciones del módulo
importlib.reload(taxi_cleaning)

# Aplicar todas las transformaciones
df_prepared = taxi_cleaning.prepare_taxi_data(df_raw)

df_prepared = taxi_cleaning.add_advanced_quality_flags(
    df_prepared
)

# Seleccionar registros de 2024
df_silver = taxi_cleaning.filter_taxi_2024(
    df_prepared
)

# Validar indicadores avanzados
validation = df_prepared.agg(
    F.count("*").alias("total"),

    F.sum(
        F.col("flag_speed_over_80").cast("long")
    ).alias("extreme_speed"),

    F.sum(
        F.col("flag_duration_over_4h").cast("long")
    ).alias("long_duration"),

    F.sum(
        F.col("flag_negative_fare").cast("long")
    ).alias("negative_fare"),

    F.sum(
        F.col("flag_negative_total").cast("long")
    ).alias("negative_total"),

    F.sum(
        F.col("flag_negative_tip").cast("long")
    ).alias("negative_tip")
).first()

assert validation["total"] == 41_169_720
assert validation["extreme_speed"] == 13_244
assert validation["long_duration"] == 23_374
assert validation["negative_fare"] == 731_024
assert validation["negative_total"] == 609_344
assert validation["negative_tip"] == 1_331

print("Validaciones avanzadas correctas.")

Validaciones avanzadas correctas.


## 6. Persistencia incremental de la capa Silver

### 6.1 Justificación

La primera escritura de los **41 millones de filas** como una sola operación produjo una interrupción de la conexión Py4J/JVM. La causa raíz no quedó demostrada; se comprobó después que Docker podía escribir un Parquet pequeño y un archivo mensual completo.

Se adoptó por ello una escritura **incremental por archivo de origen**. Cada mes se lee, transforma y agrega a un directorio temporal, en formato **Parquet/Snappy**, particionado por **`pickup_month`**. Esto no equivale a hacer una transacción ACID: por ello la salida debe validarse antes de promocionarse.

**Precauciones:**

- `source_month` identifica el archivo; `pickup_month` identifica el mes real del viaje. Algunos archivos contribuyen a más de una partición.
- `append` no deduplica. Si el staging ya existe, no se debe volver a agregar el mismo mes automáticamente.
- Una escritura interrumpida puede dejar archivos parciales. **No** se borran ni reemplazan datos existentes sin inspección.
- Si el staging existe, se reutiliza para validación. Para reconstruirlo, se debe archivar o eliminar **expresamente** la copia anterior tras verificar qué contiene.

In [12]:
from pathlib import Path
import importlib
from pyspark.sql import functions as F

importlib.reload(taxi_cleaning)

STAGING_PATH = (
    PROJECT_ROOT / "data" / "processed" / "taxi" / "_silver_2024_staging"
)

assert len(taxi_files) == 12, "Se esperaban 12 archivos de entrada."

if STAGING_PATH.exists():
    # Reejecución segura: no volver a anexar meses al mismo directorio.
    print(f"Staging existente: {STAGING_PATH}")
    print("Se omite la escritura. La siguiente celda validará su contenido.")
else:
    # Reprocesamiento por lotes mensuales, con volumen acotado por acción.
    for i, path in enumerate(taxi_files):
        source_month = path.stem.replace("yellow_tripdata_", "")
        print(f"Procesando {source_month}...", flush=True)

        # Procedencia explícita: confiable para la lectura de un único archivo.
        df_month_raw = (
            spark.read.parquet(str(path))
            .withColumn("source_month", F.lit(source_month))
        )

        df_month = taxi_cleaning.prepare_taxi_data(df_month_raw)
        df_month = taxi_cleaning.add_advanced_quality_flags(df_month)
        df_month = taxi_cleaning.filter_taxi_2024(df_month)

        # Una ruta nueva y una pasada por cada mes. La primera crea
        # la carpeta; las demás añaden Parquet al mismo destino.
        mode = "overwrite" if i == 0 else "append"
        (
            df_month.write
            .mode(mode)
            .option("compression", "snappy")
            .option("maxRecordsPerFile", 500_000)
            .partitionBy("pickup_month")
            .parquet(str(STAGING_PATH))
        )

        print(f"Mes {source_month} escrito correctamente.", flush=True)

    print("Escritura incremental completada; falta validación integral.")

Staging existente: c:\Users\derec\Desktop\Proyectos\BigData_project\data\processed\taxi\_silver_2024_staging
Se omite la escritura. La siguiente celda validará su contenido.


### 6.2 Validación del staging

Se vuelve a leer el destino físico, se verifica el número esperado de filas, la existencia de **12 particiones de `pickup_month`** y la presencia de indicadores esenciales. La partición mensual corresponde al **mes del viaje**, de modo que sus conteos pueden diferir de los recuentos por archivo `source_month`.

**Resultado registrado de la ejecución original:** `41,169,664` registros y mensaje **«Todas las validaciones de Silver pasaron»**. La nueva celda de escritura anterior omite automáticamente la recomputación si encuentra el staging existente; así puede volver a ejecutarse esta validación sin generar duplicados.

In [13]:
# Leer los archivos generados
df_silver_saved = spark.read.parquet(
    str(STAGING_PATH)
)

# Validar volumen
total_saved = df_silver_saved.count()

print(f"Registros almacenados: {total_saved:,}")

assert total_saved == 41_169_664, (
    "El número de registros no coincide "
    "con el volumen esperado."
)

# Validar distribución mensual por fecha real del viaje
df_monthly_validation = (
    df_silver_saved
    .groupBy("pickup_month")
    .count()
    .orderBy("pickup_month")
)

df_monthly_validation.show(12)

# Comprobar particiones
months = [
    row["pickup_month"]
    for row in df_monthly_validation.collect()
]

assert months == [
    f"2024-{month:02d}"
    for month in range(1, 13)
]

# Verificar que los indicadores se conservaron
required_columns = [
    "is_flex_fare",
    "flag_invalid_duration",
    "flag_zero_distance",
    "flag_speed_over_80",
    "flag_negative_fare",
    "is_mobility_core"
]

assert all(
    col in df_silver_saved.columns
    for col in required_columns
)

print("Todas las validaciones de Silver pasaron.")

Registros almacenados: 41,169,664
+------------+-------+
|pickup_month|  count|
+------------+-------+
|     2024-01|2964617|
|     2024-02|3007533|
|     2024-03|3582611|
|     2024-04|3514295|
|     2024-05|3723843|
|     2024-06|3539170|
|     2024-07|3076876|
|     2024-08|2979192|
|     2024-09|3633025|
|     2024-10|3833780|
|     2024-11|3646372|
|     2024-12|3668350|
+------------+-------+

Todas las validaciones de Silver pasaron.


### 6.3 Verificaciones adicionales de trazabilidad

Comprobamos que el staging no contiene filas fuera de 2024, que mantiene los **doce meses de origen** y que conserva la variable de procedencia. Esta verificación es complementaria a los `assert` anteriores; la cobertura de zonas TLC y la reconciliación contable exhaustiva están fuera del alcance de este notebook.

In [14]:
from pyspark.sql import functions as F

traceability = df_silver_saved.agg(
    F.count("*").alias("rows_saved"),
    F.countDistinct("source_month").alias("source_months"),
    F.countDistinct("pickup_month").alias("pickup_months"),
    F.sum(F.when(F.col("flag_out_of_year"), 1).otherwise(0)).alias("out_of_year"),
    F.sum(F.when(F.col("source_month").isNull() |
                 (F.col("source_month") == ""), 1).otherwise(0)).alias("missing_source")
).first()

assert traceability["rows_saved"] == 41_169_664
assert traceability["source_months"] == 12
assert traceability["pickup_months"] == 12
assert traceability["out_of_year"] == 0
assert traceability["missing_source"] == 0

print("Trazabilidad básica validada:", traceability.asDict())

# Diferenciar distribución por origen del archivo (source_month)
# de la distribución por fecha efectiva del viaje (pickup_month).
(
    df_silver_saved.groupBy("source_month")
    .count()
    .orderBy("source_month")
    .show(12, truncate=False)
)

Trazabilidad básica validada: {'rows_saved': 41169664, 'source_months': 12, 'pickup_months': 12, 'out_of_year': 0, 'missing_source': 0}
+------------+-------+
|source_month|count  |
+------------+-------+
|2024-01     |2964609|
|2024-02     |3007524|
|2024-03     |3582626|
|2024-04     |3514285|
|2024-05     |3723826|
|2024-06     |3539187|
|2024-07     |3076900|
|2024-08     |2979181|
|2024-09     |3633027|
|2024-10     |3833770|
|2024-11     |3646366|
|2024-12     |3668363|
+------------+-------+



### 6.4 Promoción controlada de staging a Silver

Esta acción está **separada de la escritura** para no confundir un resultado parcial con un dataset publicado. Únicamente se propone mover el staging validado a `data/processed/taxi/silver` si el destino **no existe**. No elimina ni sobrescribe carpetas preexistentes.

Si `silver/` quedó de un intento fallido anterior, revísalo y archívalo o retíralo deliberadamente **antes** de la promoción. La operación de renombrado requiere estar en el mismo sistema de archivos y permisos adecuados. Ejecutar esta celda solo tras validar las anteriores.

In [16]:
SILVER_PATH = PROJECT_ROOT / "data" / "processed" / "taxi" / "silver"

# Solo se promueve después de validar en esta sesión.
assert traceability["rows_saved"] == 41_169_664
assert STAGING_PATH.is_dir(), "No existe el staging para promocionar."
assert not SILVER_PATH.exists(), (
    f"El destino {SILVER_PATH} ya existe. "
    "Revisarlo manualmente; no se sobrescribirá."
)

STAGING_PATH.rename(SILVER_PATH)

# Tras mover los archivos, la lectura debe apuntar a la ruta definitiva.
df_silver_final = spark.read.parquet(str(SILVER_PATH))
assert df_silver_final.count() == 41_169_664
print("Capa Silver promovida y verificada:", SILVER_PATH)

Capa Silver promovida y verificada: c:\Users\derec\Desktop\Proyectos\BigData_project\data\processed\taxi\silver


## 7. Conclusiones y decisiones de calidad

1. **Cobertura:** de 41,169,720 registros Raw, se conservan **41,169,664** pickups dentro de 2024; solo se excluyen los **56** registros fuera del periodo de análisis.
2. **Faltantes estructurados:** 4,091,232 observaciones presentan cinco variables ausentes simultáneamente, asociadas en este dataset con `payment_type = 0` (Flex Fare); no se imputan sin sustento.
3. **Anomalías etiquetadas:** los ceros de distancia, duraciones no positivas, velocidades extremas e importes negativos permanecen disponibles con banderas de calidad.
4. **Diferenciación de usos:** `is_candidate_mobility` e `is_mobility_core` permiten seleccionar viajes aptos para ciertos análisis sin degradar la base Silver.
5. **Reproducibilidad:** el módulo `src/etl/taxi_cleaning.py` centraliza reglas; el notebook conserva diagnóstico, métricas y evidencia de persistencia incremental.
6. **Persistencia:** la escritura por meses y la validación de **41,169,664 filas** resultaron correctas en la corrida documentada; la publicación como `silver/` requiere la promoción controlada del apartado anterior.

### Límites y trabajo posterior

- **Calidad geográfica:** comprobar pertenencia de `PULocationID` y `DOLocationID` al catálogo TLC (no basta con que sean positivos).
- **Consistencia de fechas:** fijar y documentar la interpretación de la zona horaria al integrar los viajes con los datos diarios de NOAA.
- **Calidad monetaria:** antes de estimar ingresos, investigar ajustes y registros con importes negativos.
- **Pruebas automatizadas:** ejecutar `python -m pytest tests/test_taxi_cleaning.py -v` en Docker si ese archivo de pruebas existe en el repositorio; sus resultados no forman parte del material proporcionado.
- **Integración:** continuar con `04_02` (tratamiento de datos meteorológicos NOAA) y `04_03` (join temporal).

**Fuentes:** NYC TLC, [diccionario de Yellow Taxi](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf); [página de Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page).